# Operations, Monitoring & Evidence
Covers **Stage 4.** Complete every `# TODO` in this notebook **and** in the `src/` modules it imports. 
- Each stage opens with a **sub-task checklist**
- Capture any repo-generated evidence (MLflow UI, Docker build, CI run, drift report) as screenshots/summaries **inside the notebook/report**.

**File ownership** — 
- *Provided:* `config.py`, `src/evaluate.py`. 
- *Provided to extend:* `src/monitoring.py`, `src/retrain.py`, `src/model.py` (EmbeddingExtractor). 
- *You build:* this notebook.

### 0. Setup

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import config
from src import data_prep

## **Stage 4.1 — Prediction Logging & Confidence Monitoring** <font color="red">[5 marks]</font>

- **4.1.1 — Prediction logging with timestamp [2]** — *to be done in this notebook and the report*
- **4.1.2 — Confidence monitoring reference vs current + threshold [3]** — *to be done in this notebook and the report*

**Objective:** Log predictions and watch mean confidence drift.

**Implement in:** app.py (logging) + src/monitoring.py (confidence)

**Inputs → Outputs:** served predictions → predictions.log; reference vs current → confidence drop + alert

**TODO:** log each prediction with a timestamp (4.1.1); compute mean predicted-confidence on reference vs a current batch with an alert threshold (4.1.2).

**Depends on:** Stage 3.4 completed in Model_Development_and_Tracking.ipynb ·  **Document here:** the confidence reference→current + log tail.

In [ ]:
# Run the monitoring pipeline first
!python -m src.monitoring

In [ ]:
# 4.1.1: Show prediction log tail
log_path = config.PREDICTIONS_LOG
if log_path.exists():
    lines = log_path.read_text().strip().split('\n')
    print(f'=== Prediction Log (last 5 of {len(lines)} entries) ===')
    for line in lines[-5:]:
        print(json.dumps(json.loads(line), indent=2))
else:
    print('No predictions.log yet — run a /predict request via TestClient first.')

# 4.1.2: Show confidence block from drift_summary.json
drift = json.loads((config.ARTIFACT_DIR / 'drift_summary.json').read_text())
conf = drift['confidence']
print(f'\n=== Confidence Monitoring ===')
print(f"Reference mean confidence: {conf['reference_mean']:.4f}")
print(f"Current mean confidence:   {conf['current_mean']:.4f}")
print(f"Drop:                      {conf['drop']:.4f}")
print(f"Threshold:                 {conf['threshold']}")
print(f"Drifted:                   {conf['drifted']}")

## **Stage 4.2 — Statistical Drift Detection (Evidently + PSI)** <font color="red">[8 marks]</font>

- **4.2.1 — Per-image features + Evidently DataDriftPreset [4]** — *to be done in this notebook and the report*
- **4.2.2 — PSI per feature + interpretation [4]**— *to be done in this notebook and the report*

**Objective:** Detect drift in interpretable image features.

**Implement in:** src/monitoring.py

**Inputs → Outputs:** reference vs simulated current batch → per-feature drift + PSI + HTML report

**TODO:** compute image features + run Evidently DataDriftPreset reference vs a simulated current batch (4.2.1); compute PSI per feature + interpret + save drift_report.html (4.2.2).

**Document here:** the per-feature PSI table/plot + dataset_drift flag.

In [ ]:
# 4.2.1-4.2.2: PSI bar plot per feature vs threshold
stat = drift['statistical_drift']
feature_psi = stat['feature_psi']

print('=== Statistical Drift (PSI per feature) ===')
for feat, val in feature_psi.items():
    flag = 'DRIFTED' if val > config.PSI_THRESHOLD else 'ok'
    print(f"  {feat:20s}: PSI={val:.4f}  [{flag}]")
print(f"\nDrift share: {stat['drift_share']:.2f} (threshold={stat['threshold']})")
print(f"Statistical drift detected: {stat['drifted']}")

# Bar plot
fig, ax = plt.subplots(figsize=(10, 5))
features = list(feature_psi.keys())
values = list(feature_psi.values())
colors = ['#e74c3c' if v > config.PSI_THRESHOLD else '#2ecc71' for v in values]
bars = ax.bar(features, values, color=colors)
ax.axhline(y=config.PSI_THRESHOLD, color='red', linestyle='--', label=f'PSI threshold ({config.PSI_THRESHOLD})')
ax.set_ylabel('PSI')
ax.set_title('Feature Drift: PSI per Image Feature')
ax.legend()
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9)
fig.tight_layout()
plt.savefig(config.ARTIFACT_DIR / 'feature_psi_plot.png', dpi=150)
plt.show()

# Evidently report link
report_path = config.ARTIFACT_DIR / 'drift_report.html'
if report_path.exists():
    print(f"\n✓ Evidently drift_report.html saved at: {report_path}")

## **Stage 4.3 — Embedding Drift Detection** <font color="red">[7 marks]</font>

- **4.3.1 — 512-dim embedding extraction (reference + current) [3]** — *to be done in this notebook*
- **4.3.2 — Feature-space drift quantified (PSI) + explanation [4]** — *to be done in this notebook and the report*

### How embedding drift works (conceptual scaffolding — implement these 4 steps)
Raw-pixel features can miss *semantic* drift, so also measure drift in the model's **feature space**:
1. **Feature extraction** — use the model's **penultimate layer** (the 512-dim vector before the final classifier) as an image **embedding** (`EmbeddingExtractor` in `src/model.py`). *(4.3.1)*
2. **Embedding generation** — run the reference set and the current batch through the backbone and collect the embedding vectors (save reference to `reference_embeddings.npz`). *(4.3.1)*
3. **Feature-space comparison** — reduce each embedding to a scalar **distance to the reference centroid** (mean reference embedding) → one distribution per batch. *(4.3.2)*
4. **Drift calculation** — compute **PSI** between the reference and current distance distributions; PSI above ~0.10 signals embedding drift even when raw pixels look similar. *(4.3.2)*

**TODO:** implement steps 1–4 in `src/model.EmbeddingExtractor` + `src/monitoring.py`; report the embedding PSI and whether it exceeds the threshold.

**Document here:** the embedding PSI + a one-line interpretation of why embedding drift differs from feature drift.

In [ ]:
# 4.3.1-4.3.2: Embedding drift results
emb = drift['embedding_drift']
print('=== Embedding Drift ===')
print(f"Distance-to-centroid PSI: {emb['psi']:.4f}")
print(f"Threshold:               {emb['threshold']}")
print(f"Embedding drift:         {emb['drifted']}")

print('\n=== Interpretation ===')
print('Embedding drift captures semantic shifts in the model\'s learned feature space.')
print('While statistical feature drift (brightness, contrast, etc.) detects low-level')
print('pixel-space changes, embedding drift detects higher-level distributional shifts')
print('that may affect the model\'s decision boundary even when raw image statistics')
print('appear stable.')

# Show reference embeddings info
ref_embed = np.load(config.REFERENCE_EMBED)
print(f'\nReference embeddings shape: {ref_embed["embeddings"].shape}')
print(f'Embedding dimensionality: {ref_embed["embeddings"].shape[1]} (ResNet18 penultimate layer)')

## **Stage 4.4 — Retraining Workflow & Triggers** <font color="red">[6 marks]</font>

- **4.4.1 — Measurable retraining triggers [2]** — *to be done in this notebook and the report*
- **4.4.2 — Drift-triggered retraining + new version registered [4]** — *to be done in this notebook and the report*

**Objective:** Retrain a candidate when drift fires.

**Implement in:** src/retrain.py

**Inputs → Outputs:** drift signals → drift-augmented candidate model + new registry version

**TODO:** define multi-signal triggers (drift share / embedding PSI / confidence drop) (4.4.1); on a trigger, train a drift-augmented candidate + register a new version (4.4.2).

**Document here:** the trigger decision + candidate training summary.

In [ ]:
# 4.4.1: Display retraining triggers
print('=== Retraining Triggers ===')
print(f"Statistical drift:  {drift['statistical_drift']['drifted']}")
print(f"  - Drift share: {drift['statistical_drift']['drift_share']:.2f} > {drift['statistical_drift']['threshold']}")
print(f"Embedding drift:    {drift['embedding_drift']['drifted']}")
print(f"  - PSI: {drift['embedding_drift']['psi']:.4f} > {drift['embedding_drift']['threshold']}")
print(f"Confidence drift:   {drift['confidence']['drifted']}")
print(f"  - Drop: {drift['confidence']['drop']:.4f} > {drift['confidence']['threshold']}")
print(f"\n→ Retrain recommended: {drift['retrain_recommended']}")

In [ ]:
# 4.4.2: Run retraining
!python -m src.retrain

In [ ]:
# Load retraining decision
decision = json.loads((config.ARTIFACT_DIR / 'retraining_decision.json').read_text())
print('=== Retraining Decision ===')
print(json.dumps(decision, indent=2))

## **Stage 4.5 — Version Comparison & Rollback Governance** <font color="red">[4 marks]</font>

- **4.5.1 — Candidate vs production comparison on drifted batch [2]** — *to be done in this notebook and the report*
- **4.5.2 — Promote-on-improvement-else-rollback gate + decision recorded [2]** — *to be done in this notebook and the report*

**Objective:** Promote only on improvement; otherwise roll back.

**Implement in:** src/retrain.py

**Inputs → Outputs:** candidate vs production on a drifted batch → promote (alias move) OR rollback

**TODO:** compare candidate vs production F1 on the drifted batch (4.5.1); promote to @production only if candidate ≥ prod + epsilon else keep the incumbent + record the decision/version history (4.5.2).

**Document here:** the comparison numbers + the promote/rollback decision.

In [ ]:
# 4.5.1-4.5.2: Version comparison + promote/rollback decision
print('=== Version Comparison ===')
print(f"Production F1 (drifted): {decision.get('production_f1_drifted', 'N/A')}")
print(f"Candidate F1 (drifted):  {decision.get('candidate_f1_drifted', 'N/A')}")
print(f"Promote epsilon:         {decision.get('promote_epsilon', 'N/A')}")
print(f"\nAction: {decision['action'].upper()}")

if decision['action'] == 'promote':
    print(f"New production version: v{decision.get('candidate_version', '?')}")
else:
    print('Incumbent production model retained.')

# Show current registry state
import mlflow
from mlflow import MlflowClient
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
client = MlflowClient()

try:
    mv = client.get_model_version_by_alias(config.REGISTERED_MODEL, config.PRODUCTION_ALIAS)
    print(f"\n=== Current Production Model ===")
    print(f"Model: {config.REGISTERED_MODEL}")
    print(f"Version: v{mv.version}")
    print(f"Alias: @{config.PRODUCTION_ALIAS}")
    
    versions = client.search_model_versions(f"name='{config.REGISTERED_MODEL}'")
    print(f"\n=== Full Version History ===")
    for v in versions:
        aliases = v.aliases if hasattr(v, 'aliases') else []
        print(f"  v{v.version} — aliases={aliases}")
except Exception as e:
    print(f"Registry query failed: {e}")